In [17]:
import os
import json
import glob
import pandas as pd

In [18]:

RESULTS_DIR = "algorithm_project_ayazhan/results/results_experiment_experiment_sample_30k"  # directory with part-* files
PARAMS_PATH = "algorithm_project_ayazhan/results/results_experiment_experiment_sample_30k.json"  # parameters JSON

In [25]:
# Load metadata (id, title, authors) from same source as find_similarity
METADATA_JSON = "data/arxiv-metadata-oai-snapshot.json"
SAMPLE_SIZE = 30000  # must match find_similarity's sample_size

meta_data = []
with open(METADATA_JSON, "r") as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_SIZE:
            break
        doc = json.loads(line)
        meta_data.append({"id": doc["id"], "title": doc.get("title", ""), "authors": doc.get("authors", "")})
df_meta = pd.DataFrame(meta_data)
meta_dict = df_meta.set_index("id").to_dict("index")
print("Metadata rows:", len(meta_dict))

Metadata rows: 30000


In [34]:
def get_results_from_file(results_dir):
    """Load similar pairs from find_similarity output (one JSON line per (id1, id2, sim))."""
    tuples_list = []
    part_files = glob.glob(os.path.join(results_dir, "part-*"))
    for path in part_files:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                t = json.loads(line)
                tuples_list.append(t)
    return tuples_list

def get_parameters_from_file(params_path):
    with open(params_path, "r") as f:
        return json.load(f)

In [35]:
# Load experiment results and remap to human-readable table

results_mapped = []
if not os.path.isdir(RESULTS_DIR):
    print("Skip (no results):", RESULTS_DIR)
else:
    params = get_parameters_from_file(PARAMS_PATH)
    pairs = get_results_from_file(RESULTS_DIR)
    for (id1, id2, sim) in pairs:
        m1 = meta_dict.get(id1, {})
        m2 = meta_dict.get(id2, {})
        results_mapped.append({
            "id1": id1,
            "title1": m1.get("title", ""),
            "authors1": m1.get("authors", ""),
            "id2": id2,
            "title2": m2.get("title", ""),
            "authors2": m2.get("authors", ""),
            "jaccard": sim,
        })

df_results = pd.DataFrame(results_mapped)
print("Total similar pairs:", len(df_results))
df_results.head(10)

Skip (no results): algorithm_project_ayazhan/results/results_experiment_experiment_sample_30k
Total similar pairs: 0


""


In [36]:
# Display highest-similarity pairs for inspection

if len(df_results) > 0:
    df_sorted = df_results.sort_values("jaccard", ascending=False)
    display(df_sorted.head(20))
else:
    print("No similar pairs in results. Run find_similarity.ipynb first.")

No similar pairs in results. Run find_similarity.ipynb first.


In [37]:
# Parameters and pair counts (no proxy baseline here)

if os.path.isfile(PARAMS_PATH):
    params = get_parameters_from_file(PARAMS_PATH)
    print("Parameters:", params)
    print("Similar pairs:", len(results_mapped))

In [38]:
# Summary statistics and visualizations
if len(df_results) == 0:
    print("No similar pairs to visualize. Run find_similarity.ipynb first.")
else:
    # 1. Summary statistics
    print("=== Jaccard Similarity Summary ===")
    print(df_results["jaccard"].describe().to_string())
    print(f"\nTotal similar pairs: {len(df_results)}")

    # 2. Figure with multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # 2a. Histogram: distribution of similarity scores
    ax1 = axes[0, 0]
    ax1.hist(df_results["jaccard"], bins=15, edgecolor="black", alpha=0.7)
    ax1.set_xlabel("Jaccard similarity")
    ax1.set_ylabel("Number of pairs")
    ax1.set_title("Distribution of Similarity Scores")
    ax1.axvline(df_results["jaccard"].mean(), color="red", linestyle="--", label=f"Mean: {df_results['jaccard'].mean():.2f}")
    ax1.legend()

    # 2b. Bar chart: similarity buckets (0.5-0.6, 0.6-0.7, etc.)
    ax2 = axes[0, 1]
    bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
    labels = ["0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0"]
    bucket_counts = pd.cut(df_results["jaccard"], bins=bins, labels=labels, include_lowest=True).value_counts().reindex(labels, fill_value=0)
    bucket_counts.plot(kind="bar", ax=ax2, color="steelblue", edgecolor="black")
    ax2.set_xlabel("Similarity range")
    ax2.set_ylabel("Number of pairs")
    ax2.set_title("Pairs by Similarity Bucket")
    ax2.tick_params(axis="x", rotation=0)

    # 2c. Top authors by number of similar pairs (authors appear in id1 or id2)
    ax3 = axes[1, 0]
    author_counts = {}
    for _, row in df_results.iterrows():
        for auth_str in [row["authors1"], row["authors2"]]:
            if pd.isna(auth_str) or not auth_str:
                continue
            # Take first author if comma-separated
            first_author = str(auth_str).split(",")[0].strip()
            if first_author:
                author_counts[first_author] = author_counts.get(first_author, 0) + 1
    top_authors = pd.Series(author_counts).sort_values(ascending=True).tail(10)
    if len(top_authors) > 0:
        top_authors.plot(kind="barh", ax=ax3, color="coral", edgecolor="black")
        ax3.set_xlabel("Number of similar pairs")
        ax3.set_ylabel("Author (first author)")
        ax3.set_title("Top Authors by Similar Pair Count")
    else:
        ax3.text(0.5, 0.5, "No author data", ha="center", va="center")

    # 2d. Box plot: similarity distribution (if enough data)
    ax4 = axes[1, 1]
    ax4.boxplot(df_results["jaccard"], vert=True)
    ax4.set_ylabel("Jaccard similarity")
    ax4.set_title("Similarity Score Distribution")
    ax4.set_xticklabels(["All pairs"])

    plt.tight_layout()
    plt.show()

No similar pairs to visualize. Run find_similarity.ipynb first.
